In [21]:
import sys, pathlib, fitz
import re
import cleaning
import pandas as pd

In [22]:
#%pip install pymupdf

In [23]:
pdf_file = "../../1-1 Review Standards/Standards/(content) IEC 62304_2006.pdf"
doc = fitz.open(pdf_file) # open a document

In [108]:
def is_starting_process(s):
    return s.strip().isdigit()

def is_starting_activity(s):
    pattern = re.compile(r'^\d{1,2}\.\d{1,2}$')
    if pattern.search(s.strip()):
        return True
    else:
        return False

In [109]:
column_names = ["Process_ID", "Process", "Activity_ID", "Activity"]
df = pd.DataFrame(columns=column_names)

procedure_pattern = r'^([\s\S]*?)\s?\.{2,}\s+\d+$'

reviewing_process_id = ""
reviewing_activity = ""
hold_previuos_process = False
hold_previuos_activity = False

for page in doc:
    for line in page.get_text("text").split("\n"):
        line = line.strip()
        if hold_previuos_process:
            pattern = re.compile(procedure_pattern)
            matches = pattern.findall(line)
            #print(str(reviewing_process), " ", matches[0])
            reviewing_process_title = matches[0]
            hold_previuos_process = False
            
        if hold_previuos_activity:
            pattern = re.compile(procedure_pattern)
            matches = pattern.findall(line)
            new_row = pd.DataFrame([{"Process_ID":reviewing_process_id, "Process":reviewing_process_title.strip(), "Activity_ID":reviewing_activity, "Activity":matches[0].strip()}])
            df = pd.concat([df, new_row], ignore_index=True)
            hold_previuos_activity = False

        if is_starting_process(line):
            reviewing_process_id = line
            if int(reviewing_process_id)<5:
                continue
            hold_previuos_process = True
        else:
            hold_previuos_process = False


        if is_starting_activity(line):
            reviewing_activity = line
            if int(reviewing_process_id)<5:
                continue      
            #print(reviewing_activity)      
            hold_previuos_activity = True
        else:
            hold_previuos_activity = False


In [110]:
display(df)

,Process_ID,Process,Activity_ID,Activity
0,5,Software development PROCESS,5.1,* Software development planning
1,5,Software development PROCESS,5.2,* Software requirements analysis
2,5,Software development PROCESS,5.3,* Software ARCHITECTURAL design
3,5,Software development PROCESS,5.4,* Software detailed design
4,5,Software development PROCESS,5.5,* SOFTWARE UNIT implementation and verification
5,5,Software development PROCESS,5.6,* Software integration and integration testing
6,5,Software development PROCESS,5.7,* SOFTWARE SYSTEM testing
7,5,Software development PROCESS,5.8,* Software release
8,6,Software maintenance PROCESS,6.1,* Establish software maintenance plan
9,6,Software maintenance PROCESS,6.2,* Problem and modification analysis


In [10]:
pdf_file = "../../1-1 Review Standards/Standards/processes from 62304 (development process).pdf"
doc = fitz.open(pdf_file) # open a document

In [11]:
# Clearning footers and headers
def remove_footer_v1(page):
    # where the page number is first
    pattern = r'^(\d{1,2}\s)\n© 2006 Association for the Advancement of Medical Instrumentation ■ ANSI/AAMI/IEC 62304:2006 (\n)'
    return re.sub(re.compile(pattern), "", page.strip())
    
def remove_footer_v2(page):
    # where the page number is at the end
    pattern = r'^© 2006 Association for the Advancement of Medical Instrumentation ■ ANSI/AAMI/IEC 62304:2006 (\n)(\d{1,2}\s)\n'
    return re.sub(re.compile(pattern), "", page.strip())

# Define the list of methods for cleaning
methods = [remove_footer_v1, remove_footer_v2]

In [12]:
is_development_process = df["Process_ID"]=="5"
development_activities = df[is_development_process]

In [13]:
development_activities

,Process_ID,Process,Activity_ID,Activity
0,5,Software development PROCESS,5.1,* Software development planning
1,5,Software development PROCESS,5.2,* Software requirements analysis
2,5,Software development PROCESS,5.3,* Software ARCHITECTURAL design
3,5,Software development PROCESS,5.4,* Software detailed design
4,5,Software development PROCESS,5.5,* SOFTWARE UNIT implementation and verification
5,5,Software development PROCESS,5.6,* Software integration and integration testing
6,5,Software development PROCESS,5.7,* SOFTWARE SYSTEM testing
7,5,Software development PROCESS,5.8,* Software release


In [16]:
def is_task_title(s):
    #task_pattern = r'^(\d\.\d\.\d{1,2})\s+\n?$'
    # for the 

    pattern = r'^(\d\.\d\.\d{1,2})\s?\n?$'
    if re.search(pattern, s):
        return True
    else:
        return False

In [17]:
# Select the columns from Development Process
column_names = ["Process_ID", "Process_Name", "Activity_ID", "Activity_Name", "Task_ID", "Task_Name", "Task_Description"]
df_62304 = pd.DataFrame(columns=column_names)

# Inicialize important variables
next_index_activity = 0
column_name_activity = "Activity"
column_name_activity_id = "Activity_ID"
tasks_per_activity = []
new_row_activity = {}
new_row_task = {}

# Variables to control extraction of tasks in current activities
current_activity_name = development_activities.loc[0, column_name_activity]
current_activity_id = development_activities.loc[0, column_name_activity_id]
current_process_name = development_activities.loc[0, "Process"]
current_process_id = development_activities.loc[0, "Process_ID"]
activity_found = False

reviewing_task_id = ""
next_activity_name = ""
hold_previuos_task = False
is_exist_for = False

#start looking for the section.   
for index_page in range(0, doc.page_count):
    # Remove the footer of the page. 
    method = methods[index_page % len(methods)]  # Use modulo to cycle through methods
    page = doc[index_page].get_text("text")
    new_page = method(page)

    # Find the activities
    lines = new_page.split("\n")
    
    for index_line in range(0, len(lines)):
        
        next_index_line = index_line + 1
        line = lines[index_line].strip()
        
        if index_page == 0 and line in current_activity_name and current_activity_id in lines[next_index_line]:
                
                if next_index_activity+1 < len(development_activities):
                    is_last_activity = False
                    next_index_activity = next_index_activity+1
                    next_activity_name = development_activities.loc[next_index_activity, column_name_activity]
                    next_activity_id = development_activities.loc[next_index_activity, column_name_activity_id]
                else:
                    is_last_activity = True

                new_row_activity = {
                    "Process_ID":current_process_id, 
                    "Process_Name":current_process_name, 
                    "Activity_ID":current_activity_id, 
                    "Activity_Name":current_activity_name
                    }

                # if not is_last_activity:
                #     print(f"The specific activity '{current_activity_name}' with ID '{current_activity_id}' is found at index {index_line}. Next activity to look for is '{next_activity_name}'")
                # else:
                #     print(f"The specific activity '{current_activity_name}' with ID '{current_activity_id}' is found at index {index_line}.")
    
        else:

            # Make sure it is not the next activity
            if line in next_activity_name and next_activity_id in lines[next_index_line]:
                
                current_activity_id = next_activity_id
                current_activity_name = next_activity_name
                
                if next_index_activity+1 < len(development_activities):
                    next_index_activity = next_index_activity+1
                    next_activity_name = development_activities.loc[next_index_activity, column_name_activity]
                    next_activity_id = development_activities.loc[next_index_activity, column_name_activity_id]
               
                new_row_activity = {
                    "Process_ID":current_process_id, 
                    "Process_Name":current_process_name, 
                    "Activity_ID":current_activity_id, 
                    "Activity_Name":current_activity_name
                    }
                
                #if not is_last_activity:
                #    print(f"The specific activity '{current_activity_name}' with ID '{current_activity_id}' is found at index {index_line}. Next activity to look for is '{next_activity_name}'")
                #else:
                #    print(f"The specific activity '{current_activity_name}' with ID '{current_activity_id}' is found at index {index_line}. Last activity of the Development process")
                            
            else:
                
                if line.strip() == "":
                    continue

                if is_task_title(line):
                    # check if the line contains the format expected

                    if reviewing_task_id != "" and reviewing_task_id != line:
                        new_row = pd.DataFrame([new_row_task])
                        df_62304 = pd.concat([df_62304, new_row], ignore_index=True)

                    if int(line.strip()[0]) > 5:
                        is_exist_for = True
                        hold_previuos_task = False
                        break

                    reviewing_task_id = line
                    new_row_task = {}
                    
                    hold_previuos_task = True
                    is_title_coming = True
                    task_name = None
                    task_description = ""
                
                elif hold_previuos_task:

                    if is_title_coming:
                        task_name = line
                        is_title_coming = False
                        new_row_task = new_row_activity.copy()

                    else:
                        task_description = task_description + " " + line

                    new_row_task["Task_ID"] = reviewing_task_id
                    new_row_task["Task_Name"] = task_name
                    new_row_task["Task_Description"] = task_description
                    
                    reviewing_process_title = matches[0]
                    #print(f"Task title of activity {current_activity_name} is {task_name} with ID {reviewing_task_id}. Description of task: {task_description}")
                    
                
    if is_exist_for:
        break

                    

In [18]:
df_62304.head()

,Process_ID,Process_Name,Activity_ID,Activity_Name,Task_ID,Task_Name,Task_Description
0,5,Software development PROCESS,5.1,* Software development planning,5.1.1,Software development plan,The MANUFACTURER shall establish a software d...
1,5,Software development PROCESS,5.1,* Software development planning,5.1.2,Keep software development plan updated,The MANUFACTURER shall update the plan as dev...
2,5,Software development PROCESS,5.1,* Software development planning,5.1.3,Software development plan reference to SYSTEM ...,"a) As inputs for software development, SYSTEM..."
3,5,Software development PROCESS,5.1,* Software development planning,5.1.4,"Software development standards, methods and to...",The MANUFACTURER shall include or reference i...
4,5,Software development PROCESS,5.1,* Software development planning,5.1.5,Software integration and integration testing p...,The MANUFACTURER shall include or reference i...


In [42]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = 'output-62304_v2.xlsx'

# Export the DataFrame to Excel
df_62304.to_excel(excel_file_path, index=False)

### Get device class

In [44]:
pdf_file_path = 'output-62304_V2.xlsx'
df = pd.read_excel(pdf_file_path)
df.head()

,Process_ID,Process_Name,Activity_ID,Activity_Name,Task_ID,Task_Name,Task_Description
0,5,Software development PROCESS,5.1,* Software development planning,5.1.1,Software development plan,The MANUFACTURER shall establish a software d...
1,5,Software development PROCESS,5.1,* Software development planning,5.1.2,Keep software development plan updated,The MANUFACTURER shall update the plan as dev...
2,5,Software development PROCESS,5.1,* Software development planning,5.1.3,Software development plan reference to SYSTEM ...,"a) As inputs for software development, SYSTEM..."
3,5,Software development PROCESS,5.1,* Software development planning,5.1.4,"Software development standards, methods and to...",The MANUFACTURER shall include or reference i...
4,5,Software development PROCESS,5.1,* Software development planning,5.1.5,Software integration and integration testing p...,The MANUFACTURER shall include or reference i...


In [45]:
pattern = r'\[Class [ABC](?:, [ABC])*]'
df["Software_Class"] = ""

for index, row in df.iterrows():

    description = df.loc[index, "Task_Description"]

    match = re.search(pattern, description)

    if match:
        df.loc[index,"Software_Class"] = match.group()


In [46]:
df.head()

,Process_ID,Process_Name,Activity_ID,Activity_Name,Task_ID,Task_Name,Task_Description,Software_Class
0,5,Software development PROCESS,5.1,* Software development planning,5.1.1,Software development plan,The MANUFACTURER shall establish a software d...,"[Class A, B, C]"
1,5,Software development PROCESS,5.1,* Software development planning,5.1.2,Keep software development plan updated,The MANUFACTURER shall update the plan as dev...,"[Class A, B, C]"
2,5,Software development PROCESS,5.1,* Software development planning,5.1.3,Software development plan reference to SYSTEM ...,"a) As inputs for software development, SYSTEM...","[Class A, B, C]"
3,5,Software development PROCESS,5.1,* Software development planning,5.1.4,"Software development standards, methods and to...",The MANUFACTURER shall include or reference i...,[Class C]
4,5,Software development PROCESS,5.1,* Software development planning,5.1.5,Software integration and integration testing p...,The MANUFACTURER shall include or reference i...,"[Class B, C]"


In [47]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = 'output-62304_v3.xlsx'

# Export the DataFrame to Excel
df.to_excel(excel_file_path, index=False)